# Setup e senadores - Senado/Brasil

Monta a coorte histórica da 57ª legislatura (01/02/2023 a 31/01/2027).

A lista da legislatura traz histórico de mandatos. Por isso esta versão não considera
todo parlamentar retornado automaticamente: ela mantém somente exercícios que realmente
intersectam a 57ª legislatura.

Saídas principais:
- `senadores.csv`
- `senadores_exercicios.csv`
- `_senadores_exercicios_brasil.csv`
- `_senadores_excluidos_leg57.csv`


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "senado"
LEGISLATURA = 57

LEG_INICIO = pd.Timestamp("2023-02-01")
LEG_FIM = pd.Timestamp("2027-01-31")

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
LEGIS_BASE = "https://legis.senado.leg.br/dadosabertos"


In [ ]:
# volume e pastas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCHEMA}.{VOLUME}")

for uf in UFS:
    pasta = ROOT / uf.lower()
    pasta.mkdir(parents=True, exist_ok=True)

    teste = pasta / "_teste_escrita.tmp"
    teste.write_text("ok", encoding="utf-8")
    teste.unlink()

print("Pastas prontas:", len(UFS))


In [ ]:
# sessão HTTP
retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "PI-II-Univesp-Bronze-Senado-Brasil/2.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def lista(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def data_senado(valor):
    if valor in (None, ""):
        return pd.NaT

    texto = str(valor).strip()

    # a API usa ISO na maior parte dos endpoints
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        return pd.to_datetime(
            texto,
            format="%Y-%m-%d",
            errors="coerce",
        )

    # fallback para formatos antigos
    return pd.to_datetime(
        texto,
        errors="coerce",
        dayfirst=True,
    )

def numero_leg(obj):
    if not isinstance(obj, dict):
        return ""
    return str(obj.get("NumeroLegislatura", "")).strip()

def mandatos_do_parlamentar(parlamentar):
    if not isinstance(parlamentar, dict):
        return []

    multi = parlamentar.get("Mandatos", {})
    if isinstance(multi, dict) and "Mandato" in multi:
        return lista(multi.get("Mandato"))

    if "Mandato" in parlamentar:
        return lista(parlamentar.get("Mandato"))

    return []

def mandato_toca_leg57(mandato):
    if not isinstance(mandato, dict):
        return False

    primeira = numero_leg(mandato.get("PrimeiraLegislaturaDoMandato"))
    segunda = numero_leg(mandato.get("SegundaLegislaturaDoMandato"))

    return str(LEGISLATURA) in {primeira, segunda}

def exercicios_do_mandato(mandato):
    bloco = mandato.get("Exercicios", {}) if isinstance(mandato, dict) else {}

    if isinstance(bloco, dict):
        return lista(bloco.get("Exercicio"))

    return []

def identificacao(parlamentar):
    return parlamentar.get("IdentificacaoParlamentar", {}) or {}

def periodo_intersecta_leg57(inicio, fim):
    if pd.isna(inicio):
        return False

    fim_cmp = LEG_FIM if pd.isna(fim) else fim

    return inicio <= LEG_FIM and fim_cmp >= LEG_INICIO

def extrair_exercicios_leg57(parlamentar, fonte):
    ident = identificacao(parlamentar)
    codigo = str(ident.get("CodigoParlamentar", "")).strip()

    if not codigo:
        return []

    nome = str(
        ident.get("NomeParlamentar")
        or parlamentar.get("NomeCompletoParlamentar")
        or ""
    ).strip()

    partido = str(ident.get("SiglaPartidoParlamentar", "")).strip()
    uf_ident = str(ident.get("UfParlamentar", "")).strip().upper()

    rows = []

    for mandato in mandatos_do_parlamentar(parlamentar):
        if not mandato_toca_leg57(mandato):
            continue

        uf = str(mandato.get("UfParlamentar") or uf_ident).strip().upper()
        participacao = str(mandato.get("DescricaoParticipacao", "")).strip()
        codigo_mandato = str(mandato.get("CodigoMandato", "")).strip()

        for exercicio in exercicios_do_mandato(mandato):
            if not isinstance(exercicio, dict):
                continue

            inicio = data_senado(exercicio.get("DataInicio"))
            fim = data_senado(exercicio.get("DataFim"))

            if not periodo_intersecta_leg57(inicio, fim):
                continue

            rows.append({
                "codigo_parlamentar": codigo,
                "nome_parlamentar": nome,
                "partido": partido,
                "uf": uf,
                "codigo_mandato": codigo_mandato,
                "participacao": participacao,
                "codigo_exercicio": str(
                    exercicio.get("CodigoExercicio", "")
                ).strip(),
                "data_inicio_exercicio": (
                    inicio.date().isoformat() if not pd.isna(inicio) else ""
                ),
                "data_fim_exercicio": (
                    fim.date().isoformat() if not pd.isna(fim) else ""
                ),
                "sigla_causa_afastamento": str(
                    exercicio.get("SiglaCausaAfastamento", "")
                ).strip(),
                "descricao_causa_afastamento": str(
                    exercicio.get("DescricaoCausaAfastamento", "")
                ).strip(),
                "fonte": fonte,
            })

    return rows

def parlamentares_legislatura(payload):
    return lista(
        payload
        .get("ListaParlamentarLegislatura", {})
        .get("Parlamentares", {})
        .get("Parlamentar")
    )

def parlamentares_atuais(payload):
    return lista(
        payload
        .get("ListaParlamentarEmExercicio", {})
        .get("Parlamentares", {})
        .get("Parlamentar")
    )


In [ ]:
# lista da legislatura + lista atual
url_leg = f"{LEGIS_BASE}/senador/lista/legislatura/{LEGISLATURA}.json"

r = session.get(
    url_leg,
    params={"exercicio": "S"},
    timeout=(30, 180),
)
r.raise_for_status()
parlamentares_leg = parlamentares_legislatura(r.json())

r = session.get(
    f"{LEGIS_BASE}/senador/lista/atual.json",
    timeout=(30, 180),
)
r.raise_for_status()
parlamentares_atual = parlamentares_atuais(r.json())

print("Retornados pela legislatura:", len(parlamentares_leg))
print("Em exercício atualmente:", len(parlamentares_atual))

exercicios_rows = []

for p in parlamentares_leg:
    exercicios_rows.extend(
        extrair_exercicios_leg57(p, "lista_legislatura_57")
    )

# a lista atual entra como verificação/complemento
for p in parlamentares_atual:
    exercicios_rows.extend(
        extrair_exercicios_leg57(p, "lista_atual")
    )

if not exercicios_rows:
    raise RuntimeError(
        "Nenhum exercício real da 57ª legislatura foi encontrado."
    )

exercicios = pd.DataFrame(exercicios_rows)

# a mesma relação pode vir das duas fontes
chaves_exercicio = [
    "codigo_parlamentar",
    "uf",
    "codigo_mandato",
    "codigo_exercicio",
    "data_inicio_exercicio",
    "data_fim_exercicio",
]

exercicios = (
    exercicios
    .sort_values(["codigo_parlamentar", "data_inicio_exercicio", "fonte"])
    .drop_duplicates(subset=chaves_exercicio, keep="first")
    .reset_index(drop=True)
)

# só UFs válidas entram nas pastas estaduais
exercicios_validos = exercicios[exercicios["uf"].isin(UFS)].copy()

# confere se alguma data ficou invertida
inicio_chk = pd.to_datetime(
    exercicios_validos["data_inicio_exercicio"],
    errors="coerce",
)
fim_chk = pd.to_datetime(
    exercicios_validos["data_fim_exercicio"],
    errors="coerce",
)

mask_invertida = (
    inicio_chk.notna()
    & fim_chk.notna()
    & (inicio_chk > fim_chk)
)

if mask_invertida.any():
    casos = exercicios_validos.loc[
        mask_invertida,
        [
            "codigo_parlamentar",
            "nome_parlamentar",
            "data_inicio_exercicio",
            "data_fim_exercicio",
        ],
    ]

    raise RuntimeError(
        "Foram encontrados períodos de exercício com data inicial "
        "posterior à data final:\n"
        + casos.to_string(index=False)
    )

print("Exercícios válidos:", len(exercicios_validos))
print(
    "Senadores históricos únicos:",
    exercicios_validos["codigo_parlamentar"].nunique()
)


In [ ]:
# identifica quem veio no endpoint da legislatura, mas não teve exercício real na janela
codigos_validos = set(
    exercicios_validos["codigo_parlamentar"].astype(str)
)

excluidos = []

for p in parlamentares_leg:
    ident = identificacao(p)
    codigo = str(ident.get("CodigoParlamentar", "")).strip()

    if not codigo or codigo in codigos_validos:
        continue

    mandatos = [
        m for m in mandatos_do_parlamentar(p)
        if mandato_toca_leg57(m)
    ]

    datas = []

    for m in mandatos:
        for e in exercicios_do_mandato(m):
            if isinstance(e, dict):
                datas.append({
                    "inicio": e.get("DataInicio"),
                    "fim": e.get("DataFim"),
                })

    excluidos.append({
        "codigo_parlamentar": codigo,
        "nome_parlamentar": str(
            ident.get("NomeParlamentar")
            or p.get("NomeCompletoParlamentar")
            or ""
        ).strip(),
        "uf_identificacao": str(
            ident.get("UfParlamentar", "")
        ).strip().upper(),
        "motivo": "sem_exercicio_intersectando_leg57",
        "exercicios_retornados": json.dumps(
            datas,
            ensure_ascii=False,
        ),
    })

excluidos_df = pd.DataFrame(excluidos)

excluidos_df.to_csv(
    ROOT / "_senadores_excluidos_leg57.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

exercicios_validos.to_csv(
    ROOT / "_senadores_exercicios_brasil.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

print("Excluídos do universo histórico:", len(excluidos_df))

if len(excluidos_df):
    display(excluidos_df)


In [ ]:
# arquivos estaduais
resumo = []

for uf in UFS:
    ex_uf = exercicios_validos[
        exercicios_validos["uf"].eq(uf)
    ].copy()

    ex_uf.to_csv(
        ROOT / uf.lower() / "senadores_exercicios.csv",
        sep=";",
        index=False,
        encoding="utf-8",
    )

    if ex_uf.empty:
        sen_uf = pd.DataFrame(columns=[
            "codigo_parlamentar",
            "nome_parlamentar",
            "partido",
            "uf",
            "primeiro_exercicio_leg57",
            "ultimo_fim_exercicio_leg57",
            "participacoes",
        ])
    else:
        sen_uf = (
            ex_uf
            .sort_values(
                ["codigo_parlamentar", "data_inicio_exercicio"]
            )
            .groupby("codigo_parlamentar", as_index=False)
            .agg(
                nome_parlamentar=("nome_parlamentar", "last"),
                partido=("partido", "last"),
                uf=("uf", "last"),
                primeiro_exercicio_leg57=(
                    "data_inicio_exercicio", "min"
                ),
                ultimo_fim_exercicio_leg57=(
                    "data_fim_exercicio", "max"
                ),
                participacoes=(
                    "participacao",
                    lambda s: "|".join(
                        sorted(set(x for x in s if x))
                    ),
                ),
            )
        )

    sen_uf.to_csv(
        ROOT / uf.lower() / "senadores.csv",
        sep=";",
        index=False,
        encoding="utf-8",
    )

    resumo.append({
        "uf": uf,
        "senadores_historicos_unicos": (
            ex_uf["codigo_parlamentar"].nunique()
        ),
        "periodos_de_exercicio": len(ex_uf),
    })

resumo_df = pd.DataFrame(resumo)

resumo_df.to_csv(
    ROOT / "_senadores_por_uf.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

display(resumo_df)


In [ ]:
config = {
    "catalogo": CATALOGO,
    "schema": SCHEMA,
    "volume": VOLUME,
    "raiz": str(ROOT),
    "legislatura": LEGISLATURA,
    "inicio_legislatura": LEG_INICIO.date().isoformat(),
    "fim_legislatura": LEG_FIM.date().isoformat(),
    "anos": [2023, 2024, 2025, 2026],
    "ufs": UFS,
    "fonte_legislativa": LEGIS_BASE,
    "criterio_senadores": (
        "somente codigos com ao menos um periodo de exercicio "
        "que intersecta a 57a legislatura"
    ),
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
}

with (ROOT / "_config_brasil.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Setup histórico concluído.")
